In [ ]:
### import dependencies
import os
import os.path as op
from os.path import join
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import gridspec
import pickle

import mne
from mne.minimum_norm import apply_inverse, read_inverse_operator
from mne import read_evokeds

In [ ]:
#========= parameters =========#
# stc parameters
SNR = 3
method = "dSPM"
fixed = False
n_jobs = 4
lambda2 = 1.0 / SNR ** 2.0

#===========================#
expt = 'EXPT'
ROOT = f'/path/to/{expt}'
os.chdir(ROOT)

epochs_dir = join(ROOT, 'data/epochs/')
evokeds_dir = join(ROOT, 'data/evokeds/')
subjects_dir = join(ROOT,'data/mri/')
raw_dir = join(ROOT, 'data/raw/')
meg_dir = join(ROOT, 'data/meg/')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'data/stc/')

excluded = ['R0000']
subjects = [i[:5] for i in os.listdir(raw_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")]

output_dir = os.path.join(ROOT, 'figures')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
# set up the source space and roi areas to plot
fsave_vertices = [np.arange(2562), np.arange(2562)]
hemi_idx = np.arange(0, 2562, 1)

src_fname = os.path.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-4-src.fif')
src = mne.read_source_spaces(src_fname)

# brodmann areas of interest
brodmann_areas = ['17', '20', '21', '22', '38', '39', '11', '44', '45']

# define colors for conditions
colors = {
    'CONDITION1': 'tab:blue',
    'CONDITION2': 'tab:orange',
    'CONDITION3': 'tab:green',
    'CONDITION4': 'tab:red'
}

conditions = list(colors.keys())

In [ ]:
## visualize parcellation with fsaverage
annot_name = 'PALS_B12_Brodmann'
brodmann_labels = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir=subjects_dir, hemi=hemi)

areas_of_interest = ['Brodmann.' + area + '-lh'for area in brodmann_areas]
selected_labels = [
    label for label in brodmann_labels if label.name in areas_of_interest
]

for label_name in areas_of_interest:
    Brain = mne.viz.get_brain_class()
    label = [label for label in brodmann_labels if label.name == label_name][0]
    
    color = 'tab:green'
    # Brain views (lateral, ventral)
    images = []
    for view in ['lateral', 'ventral']: #, 'ventral'
        if '17' in label_name:
            if view == 'ventral':
                view = view
            else:
                view = 'medial'
        brain = Brain(
            "fsaverage",
            "rh",
            "pial",  # Use the pial surface
            subjects_dir=subjects_dir,
            cortex="low_contrast",
            background="white",
            size=(800, 600),
            views=view
        )
        
        brain.add_label(label, borders=False, color=color)
        
        screenshot = brain.screenshot()
        brain.close()  # Close to free memory

        # crop the margins and gap between hemispheres
        remove_pix = (screenshot != 255).any(-1)
        remove_row = remove_pix.any(1)
        remove_col = remove_pix.any(0)
        image = screenshot[remove_row][:, remove_col]

        images.append(np.rot90(image) if view == 'ventral' else image)

    # define figure and gridspec layout
    fig = plt.figure(figsize=(8, 8)) #8, 16
    gs = gridspec(2, 1, width_ratios=[4], height_ratios=[1, 0.5], wspace=0.3) #2,1
    
    ax_brain_lateral = fig.add_subplot(gs[0])
    ax_brain_ventral = fig.add_subplot(gs[1])

    for ax, img in zip([ax_brain_lateral, ax_brain_ventral], images): #
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    fig.savefig(f'{label.name}_rh_brain.pdf')
    plt.close()

In [ ]:
#========= parameters =========#
hemi = 'lh'
t = 901
tmin = -0.1
tmax = 0.8

# walkthrough: each brodmann area plotted
for ba_num in brodmann_areas:
    print(f"Processing Brodmann area {ba_num}")
    brodmann = mne.read_labels_from_annot("fsaverage", "PALS_B12_Brodmann", subjects_dir=subjects_dir, hemi=hemi)
    label_name = f'Brodmann.{ba_num}-{hemi}'
    label = [label for label in brodmann if label.name == label_name][0]
    
    fig = plt.figure(figsize=(5, 3))
    ax = fig.add_subplot(111)
    subject_data = []

    for subj in subjects:
        subject_conditions = []
        for cond in conditions:
            filename = os.path.join(stc_dir, cond, f'{subj}_{cond}_dSPM')
            try:
                stc = mne.read_source_estimate(filename)
                timeseries = stc.extract_label_time_course(labels=label, src=src, mode="mean_flip")
                subject_conditions.append(timeseries)
            except Exception as e:
                print(f"error processing {filename}: {e}")
                subject_conditions.append(np.zeros((1, t)))
        subject_data.append(subject_conditions)

    subject_data = np.array(subject_data)
    subj_avg = subject_data.mean(axis=0)
    if subj_avg.shape[1] != t:
        subj_avg = subj_avg.reshape((len(conditions), t))
    subject_data = subject_data.reshape((len(subjects), len(conditions), t))

    for j, cond in enumerate(conditions):
        mean_vals = subj_avg[j, :]
        std_vals = subject_data[:, j, :].std(axis=0) / np.sqrt(len(subjects))
        ax.plot(stc.times, mean_vals, label=cond, color=colors[cond], linewidth=2)
        #ax.fill_between(stc.times, mean_vals - std_vals, mean_vals + std_vals, alpha=0.2, color=colors[cond])

    ax.set_title(f'Brodmann Area {ba_num}')
    ax.legend()
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('dSPM')
    ax.set_xlim(tmin, tmax)

    fig.tight_layout()
    #fig.savefig(os.path.join(output_dir, f'BA{ba_num}-{len(subjects)}.pdf'))
    plt.show()
    plt.close(fig)